# D - Computational efficiency: PINN vs FEM (Cluster 6)

**Reviewer comments addressed**

* **R1.6 / R2.5 / R3.2** - the paper claims PINNs "accelerate" simulation but
  gives no runtime / memory evidence. For a forward problem like this, a trained
  PINN is often *slower* than FEM once training is counted.

We report, **honestly**, three numbers and let the reader judge:
`FEM solve`, `PINN training` (one-off) and `PINN inference` (per evaluation,
mesh-free). The scientifically safe framing (per the strategy) is that the PINN
benefit is **mesh-free / surrogate evaluation**, not raw speed.

In [ ]:
# === Environment setup (Colab + local) ===
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Daniel14gonc/PINNs_piezoelectricity.git'

if IN_COLAB:
    if not os.path.isdir('PINNs_piezoelectricity'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('PINNs_piezoelectricity')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-e', '.'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'scikit-fem'], check=True)
else:
    # Local: walk up from the notebook to find the repo root (the folder
    # containing src/pinn_piezo), chdir there and put src on the path.
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, 'src', 'pinn_piezo')):
            break
        d = os.path.dirname(d)
    os.chdir(d)
    if os.path.join(d, 'src') not in sys.path:
        sys.path.insert(0, os.path.join(d, 'src'))

import torch
print('cwd       :', os.getcwd())
print('torch     :', torch.__version__)
print('cuda avail:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Where this notebook writes its tables / figures.
from pathlib import Path
OUT = Path('outputs/revision'); OUT.mkdir(parents=True, exist_ok=True)
print('Artefacts ->', OUT.resolve())

In [ ]:
import time, numpy as np, pandas as pd, torch
from pinn_piezo import fem
from pinn_piezo.config import MODELS_DIR
from pinn_piezo.indirect import model as imodel, train as itrain
from pinn_piezo.indirect.train import tensorize

# --- FEM solve time (a couple of mesh resolutions) ---
fem_rows = []
for (nx, ny) in [(150, 6), (300, 10), (500, 16)]:
    r = fem.solve_piezo('indirect', nx=nx, ny=ny, voltage=100.0)
    fem_rows.append({'method': f'FEM solve (mesh {nx}x{ny}, {r.n_dofs} dofs)',
                     'time_s': r.runtime_total})
fem_df = pd.DataFrame(fem_rows); print(fem_df)

In [ ]:
# --- PINN training time (one-off cost) ---
torch.set_default_dtype(torch.float64)
arrays = itrain.load_dataset('data', suffix='_m1', fraction=1.0)
tensors = itrain.to_device(arrays, DEVICE, dtype=torch.float64)
np.random.seed(0); torch.manual_seed(0)
model = imodel.build_default_model(device=DEVICE, model_type='pyramid')

QUICK = True
import os
EP_ADAM  = int(os.environ.get('REV_ADAM',  300 if QUICK else 1000))
EP_LBFGS = int(os.environ.get('REV_LBFGS', 50  if QUICK else 200))
res = itrain.train(model, tensors, epochs_adam=EP_ADAM, epochs_lbfgs=EP_LBFGS)
train_time = res['total_time']
print(f'PINN training: {train_time:.1f} s for {EP_ADAM}+{EP_LBFGS} epochs')

# --- PINN inference time (mesh-free, per N points) ---
for N in (1_000, 10_000, 100_000):
    X = np.random.rand(N, 2) * [0.1, 1e-3]
    Xt = tensorize(X, DEVICE)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = model(Xt).detach()
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    print(f'  PINN inference {N:>7,} pts: {(time.perf_counter()-t0)*1e3:.2f} ms')

# GPU memory (if available)
if DEVICE.type == 'cuda':
    print('peak GPU mem (MB):', torch.cuda.max_memory_allocated()/1e6)

In [ ]:
# --- Assemble the honest comparison table ---
infer_X = tensorize(np.random.rand(10_000, 2) * [0.1, 1e-3], DEVICE)
if DEVICE.type == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter(); _ = model(infer_X).detach()
if DEVICE.type == 'cuda': torch.cuda.synchronize()
infer_time = time.perf_counter() - t0

summary = pd.concat([
    fem_df,
    pd.DataFrame([{'method': f'PINN training ({EP_ADAM}+{EP_LBFGS} epochs)', 'time_s': train_time},
                  {'method': 'PINN inference (10,000 pts, mesh-free)', 'time_s': infer_time}]),
], ignore_index=True)
summary.to_csv(OUT / 'efficiency_table.csv', index=False)
summary

---
### Rebuttal snippet (Cluster 6)
> *We added a runtime comparison (Table …). Training the PINN is a one-off cost
> of … s and is [slower/comparable] to a single FEM solve (… s); once trained,
> inference is mesh-free and evaluates 10⁴ arbitrary points in … ms. We have
> therefore repositioned the contribution around **mesh-free evaluation and
> surrogate-model capability** rather than raw speed, and removed unsupported
> "acceleration" claims.*

> **Note.** Do **not** force a speed advantage if one does not exist; the
> mesh-free / surrogate framing is the defensible one.